# Scenario 1b — US / Russia nuclear war with reduced trade (N, P, K)

Same production shock as scenario 1, plus a **trade shock** on all bilateral flows
touching the United States or Russian Federation:

| Group | Surviving production fraction | Surviving trade fraction (US/RU flows) |
|---|---|---|
| United States of America | 0.40 (−60 %) | 0.40 (−60 %) on any flow touching US |
| Russian Federation | 0.40 (−60 %) | 0.40 (−60 %) on any flow touching RU |
| Every other country | 0.82 (−18 %) | unchanged |

Baseline run uses historical `T0`; shocked run uses `T0_shocked`.

Outputs: `../results/` files tagged `N_scenario1_us_ru_reduced_trade_baseline` (and P, K, shocked).

> Before running: FAOSTAT bulk CSVs in `data/` — see `data/README.md`.


In [ ]:
import sys, pathlib

sys.path.append(str(pathlib.Path.cwd().parent))

import pandas as pd

from src.preprocessing import (
    load_nutrient,
    apply_shock_reported,
    apply_trade_shock_reported,
    NUTRIENT_NAMES,
)
from src.model import FertilizerRAS
from src.postprocessing import (
    build_comparison,
    most_affected,
    least_affected,
    global_summary,
    sanity_checks,
    save_result,
)

pd.set_option('display.float_format', '{:,.0f}'.format)


## 1. Configuration

In [ ]:
NUTRIENTS = ['N', 'P', 'K']
YEARS = list(range(2014, 2019))
MIN_THRESHOLD = 1_000  # tonnes

INPUTS_CSV = '../data/Inputs_FertilizersNutrient_E_All_Data/Inputs_FertilizersNutrient_E_All_Data_NOFLAG.csv'
TRADE_CSV = '../data/Fertilizers_DetailedTradeMatrix_E_All_Data/Fertilizers_DetailedTradeMatrix_E_All_Data_NOFLAG.csv'

HEAVY_HIT = {
    'United States of America': 0.40,  # -60%
    'Russian Federation': 0.40,  # -60%
}
REST_OF_WORLD_FRACTION = 0.82  # -18% for every other country
TRADE_RESTRICTED = list(HEAVY_HIT.keys())
TRADE_SURVIVING_FRACTION = 0.40  # -60% on all flows touching US/RU
SCENARIO_TAG = 'scenario1_us_ru_reduced_trade'
SCENARIO1_TAG = 'scenario1_us_ru_nuclear'  # for optional comparison

SAVE_PLOTS = False


In [ ]:
def build_scenario1_shock(country_index: pd.Index) -> dict[str, float]:
    """US/RU -60 %, rest of world -18 % (surviving fractions)."""
    shock = {country: REST_OF_WORLD_FRACTION for country in country_index}
    for country, frac in HEAVY_HIT.items():
        if country in shock:
            shock[country] = frac
        else:
            print(f'WARNING: {country!r} not in FAOSTAT index — check exact FAO name.')
    return shock


def us_ru_trade_volume(T: pd.DataFrame, countries: list[str]) -> float:
    """Sum of bilateral flows where exporter or importer is in ``countries``."""
    mask = pd.DataFrame(False, index=T.index, columns=T.columns)
    for c in countries:
        if c in T.index:
            mask.loc[c, :] = True
            mask.loc[:, c] = True
    return float((T.values * mask.values).sum())


## 2. Run scenario 1b for each nutrient

In [ ]:
all_results: dict[str, dict] = {}
cross_rows: list[dict] = []

for nutrient in NUTRIENTS:
    label = NUTRIENT_NAMES[nutrient]
    print('\n' + '=' * 72)
    print(f'=== {label} ({nutrient}) ===')
    print('=' * 72)

    data = load_nutrient(
        INPUTS_CSV, TRADE_CSV,
        nutrient=nutrient,
        years=YEARS,
        min_threshold=MIN_THRESHOLD,
    )
    P, C, T0 = data['P'], data['C'], data['T0']
    n_countries = len(data['countries'])

    print(f'Countries: {n_countries}')
    print(f'Total production: {P.sum():,.0f} t')
    print(f'Total demand:     {C.sum():,.0f} t')
    print(f'Total trade (T0): {T0.values.sum():,.0f} t')

    shock = build_scenario1_shock(P.index)
    P_shocked, report = apply_shock_reported(P, shock)
    T0_shocked, trade_report = apply_trade_shock_reported(
        T0, TRADE_RESTRICTED, TRADE_SURVIVING_FRACTION
    )

    if report.unmatched:
        print('Unmatched production shock:', report.unmatched)
    if trade_report.unmatched:
        print('Unmatched trade shock:', trade_report.unmatched)

    us_ru_t0_before = us_ru_trade_volume(T0, TRADE_RESTRICTED)
    us_ru_t0_after = us_ru_trade_volume(T0_shocked, TRADE_RESTRICTED)
    print(
        f'US/RU-touching trade (T0): {us_ru_t0_before:,.0f} t → {us_ru_t0_after:,.0f} t '
        f'({TRADE_SURVIVING_FRACTION:.0%} surviving)'
    )
    display(trade_report.as_dataframe())

    prod_before = float(P.sum())
    prod_after = float(P_shocked.sum())
    decline_pct = (1 - prod_after / prod_before) * 100
    us_ru_share = (
        P.get('United States of America', 0) + P.get('Russian Federation', 0)
    ) / max(prod_before, 1e-12) * 100

    print(f'Global production decline: {decline_pct:.2f} %')
    print(f'US+Russia share of baseline fertilizer production: {us_ru_share:.1f} %')

    baseline = FertilizerRAS(P, C, T0).run(verbose=True)
    shocked = FertilizerRAS(P_shocked, C, T0_shocked).run(verbose=True)

    comparison = build_comparison(baseline, shocked)
    gsum = global_summary(baseline, shocked)

    tag = f'{nutrient}_{SCENARIO_TAG}'
    paths_b = save_result(baseline, '../results', tag=f'{tag}_baseline')
    paths_s = save_result(shocked, '../results', tag=f'{tag}_shocked')
    print('Saved:', paths_b, paths_s)

    print('\nGlobal summary:')
    display(gsum)

    print('\nUS / Russia coverage (scenario 1b):')
    for country in TRADE_RESTRICTED:
        if country in comparison.index:
            row = comparison.loc[country]
            print(
                f"  {country}: {row['Cover_base_%']:.1f}% → "
                f"{row['Cover_shock_%']:.1f}% ({row['Change_pp']:+.1f} pp)"
            )

    print('\nMost affected (coverage drop):')
    display(most_affected(comparison, n=10))

    print('\nLeast affected:')
    display(least_affected(comparison, n=5))

    cover_base = float(baseline.F.sum() / max(baseline.C.sum(), 1e-12) * 100)
    cover_shock = float(shocked.F.sum() / max(shocked.C.sum(), 1e-12) * 100)

    cross_rows.append({
        'Nutrient': label,
        'Code': nutrient,
        'Countries': n_countries,
        'Prod_before_t': prod_before,
        'Prod_after_t': prod_after,
        'Prod_decline_%': decline_pct,
        'US_RU_prod_share_%': us_ru_share,
        'US_RU_T0_before_t': us_ru_t0_before,
        'US_RU_T0_after_t': us_ru_t0_after,
        'Prod_loss_t': prod_after - prod_before,
        'Trade_delta_t': float(shocked.X.values.sum() - baseline.X.values.sum()),
        'Cover_base_%': cover_base,
        'Cover_shock_%': cover_shock,
        'Cover_change_pp': cover_shock - cover_base,
    })

    all_results[nutrient] = {
        'baseline': baseline,
        'shocked': shocked,
        'comparison': comparison,
        'global_summary': gsum,
        'P': P,
        'C': C,
        'T0': T0,
        'T0_shocked': T0_shocked,
        'P_shocked': P_shocked,
        'report': report,
        'trade_report': trade_report,
    }


## 3. Cross-nutrient summary

In [ ]:
cross = pd.DataFrame(cross_rows).set_index('Code')
display(cross)
cross_path = pathlib.Path('../results') / f'cross_summary_{SCENARIO_TAG}.csv'
cross.to_csv(cross_path)
print(f'Wrote {cross_path}')


## 4. Comparison with scenario 1 (production shock only)

If scenario 1 result CSVs exist, compare US/RU coverage side by side.

In [ ]:
results_dir = pathlib.Path('../results')
compare_rows = []

for nutrient in NUTRIENTS:
    s1_path = results_dir / f'summary_{nutrient}_{SCENARIO1_TAG}_shocked.csv'
    s1b = all_results[nutrient]['comparison']
    if not s1_path.exists():
        print(f'Skip {nutrient}: {s1_path.name} not found (run scenario 1 first)')
        continue
    s1 = pd.read_csv(s1_path, index_col=0)
    for country in TRADE_RESTRICTED:
        if country not in s1b.index or country not in s1.index:
            continue
        compare_rows.append({
            'Nutrient': nutrient,
            'Country': country,
            'Coverage_scenario1_%': float(s1.loc[country, 'Coverage_%']),
            'Coverage_scenario1b_%': float(s1b.loc[country, 'Cover_shock_%']),
            'Delta_pp': float(s1b.loc[country, 'Cover_shock_%'])
            - float(s1.loc[country, 'Coverage_%']),
        })

if compare_rows:
    cmp_df = pd.DataFrame(compare_rows)
    display(cmp_df)
    cmp_path = results_dir / f'us_ru_coverage_compare_{SCENARIO_TAG}.csv'
    cmp_df.to_csv(cmp_path, index=False)
    print(f'Wrote {cmp_path}')


## 5. Mass-balance sanity checks

In [ ]:
for nutrient in NUTRIENTS:
    r = all_results[nutrient]
    print(f'\n=== {NUTRIENT_NAMES[nutrient]} ({nutrient}) — baseline ===')
    display(sanity_checks(r['baseline']))
    print(f'=== {NUTRIENT_NAMES[nutrient]} ({nutrient}) — shocked ===')
    display(sanity_checks(r['shocked']))
